In [2]:
import numpy as np
from iminuit import Minuit
from iminuit.cost import ExtendedBinnedNLL, ExtendedUnbinnedNLL
from scipy.stats import expon, norm, uniform, chi2
from collections.abc import Callable
import plotly.graph_objects as go
import inspect
import pandas as pd

In [31]:
def gauss_cdf(x, a, mu, sigma):
    return a * norm.cdf(x, mu, sigma)

In [9]:
data = pd.read_csv('/home/matti/uni/lab/Lab-muons/data_sim_exp_unif.txt', sep  = ' ')
print(data)

                N    A         a         b       tau         e          err_N  \
0    45193.596236  0.0  0.997711  0.000001  0.000002  0.000007    2773.012217   
1    45110.013482  0.0  0.995286  0.000002  0.000002  0.000006    2764.232933   
2    45088.816002  0.0  0.994353  0.000002  0.000002  0.000006    2756.086674   
3    45268.255836  0.0  0.999799  0.000001  0.000002  0.000007   54567.394364   
4    45057.052118  0.0  0.993289  0.000002  0.000002  0.000006  133358.682579   
..            ...  ...       ...       ...       ...       ...            ...   
195  45339.722904  0.0  1.002106  0.000001  0.000002  0.000006    2792.177274   
196  45313.644923  0.0  1.001292  0.000001  0.000002  0.000006    2786.986736   
197  45101.048333  0.0  0.994779  0.000002  0.000002  0.000006    2758.054299   
198  45174.325343  0.0  0.996675  0.000002  0.000002  0.000006    2760.526237   
199  45554.481223  0.0  1.008578  0.000001  0.000002  0.000007    2825.830812   

     err_A     err_a       

# Tau

In [ ]:
nbin = 10
count, edges = np.histogram(data['tau'], bins= nbin, density = False)

fig = go.Figure()
fig.add_trace(go.Bar(x = edges, y = count, width = np.diff(edges), name ='tau distribution'))
fig.update_layout(xaxis_title = "Time [s]", yaxis_title = "Counts")
fig.show()

In [47]:
cost = ExtendedBinnedNLL(count, edges, gauss_cdf)
m = Minuit(cost, 200, 2.085e-6, 0.3e-8)
m.fixed['a'] = True
m.migrad()

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 3.883 (χ²/ndof = 0.5)      │              Nfcn = 79               │
│ EDM = 4.05e-05 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name  │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼───────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ a     │    200    │     2     │            │            │         │         │  yes  │
│ 1 │ mu    │ 2.0904e-6 │ 0.0020e-6 │            │            │         │         │       │
│ 2 │ sigma │  27.6e-9  │  1.6e-9   │            │            │         │         │       │
└───┴───────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───────┬────────────────────────────┐
│       │        a       mu    sigma │
├───────┼────────────────────────────┤
│     a │        0    0e-18        0 │
│    mu │    0e-18 4.15e-18        0 │
│ sigma │        0        0 2.67e-18 │
└───────┴────────────────────────────┘

In [59]:
print(m.values[1])
print("distanza sigma da valore trovato con questa interpolazione e interpolazione dataset (errore somma in quadratura):", 
      abs((m.values[1] - 2.1193010893993915e-06))/np.sqrt((m.errors[1])**2 + (5.37218123526306e-08)**2) )

2.09036407663405e-06
distanza sigma da valore trovato con questa interpolazione e interpolazione dataset (errore somma in quadratura): 0.5382585999563051


# a

In [60]:
count, edges = np.histogram(data["a"], bins = 10)

fig = go.Figure()
fig.add_trace(go.Bar(x = edges, y = count, width = np.diff(edges), name ='tau distribution'))
fig.update_layout(xaxis_title = "Time [s]", yaxis_title = "Counts")
fig.show()

In [61]:
cost = ExtendedBinnedNLL(count, edges, gauss_cdf)
m = Minuit(cost, 200, 1, 5e-3)
m.fixed['a'] = True
m.migrad()

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 6.794 (χ²/ndof = 0.8)      │              Nfcn = 40               │
│ EDM = 3.63e-05 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name  │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼───────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ a     │    200    │     2     │            │            │         │         │  yes  │
│ 1 │ mu    │ 998.9e-3  │  0.5e-3   │            │            │         │         │       │
│ 2 │ sigma │  6.5e-3   │  0.4e-3   │            │            │         │         │       │
└───┴───────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───────┬────────────────────────────┐
│       │        a       mu    sigma │
├───────┼────────────────────────────┤
│     a │        0        0        0 │
│    mu │        0 2.34e-07  0.02e-6 │
│ sigma │        0  0.02e-6 1.56e-07 │
└───────┴────────────────────────────┘

In [62]:
print(m.values[1])
print("distanza sigma da valore trovato con questa interpolazione e interpolazione dataset (errore somma in quadratura):", 
      abs((m.values[1] - 1.385))/np.sqrt((m.errors[1])**2 + (0.024)**2) )

0.9988938771969841
distanza sigma da valore trovato con questa interpolazione e interpolazione dataset (errore somma in quadratura): 16.08449200550579
